# PhysChemCAL — Colab Runner

Clones the repo, installs dependencies, mounts the dataset from Google Drive, and runs the pipeline (smoke test / full training / Phase-3 counterfactuals) — see the repo's `README.md` for the full flag reference.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better), then run the cells top to bottom. Fill in `REPO_URL` and check `PKL_PATH` in the Configuration cell first.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Data (`.pkl` files) lives on Drive, never in git — see `README.md` "Data setup".

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

Fill these in once per session.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # TODO: fill in
REPO_DIR = "/content/capstone_complete"
PKL_PATH = "/content/drive/MyDrive/capstone_dataset/drugood_lbap_ec50_scaffold_3d.pkl"  # TODO: check this matches your Drive layout

## 4. Clone (or update) the repo

In [ ]:
import os
if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd $REPO_DIR

## 5. Install dependencies

`torch` is deliberately **not** installed here — Colab already ships a torch build matched to its CUDA driver, and `requirements.txt` leaves it commented out for exactly this reason (see README "Installation").

In [ ]:
!pip install -q -r requirements.txt

## 6. Sanity check: confirm the `.pkl` is where you think it is

In [ ]:
import os
assert os.path.exists(PKL_PATH), f"Can't find {PKL_PATH} -- check the Drive path / that the upload finished."
print("Found:", PKL_PATH, f"({os.path.getsize(PKL_PATH)/1e6:.1f} MB)")

## 7. Smoke test

Fast pipeline check on a truncated slice of the data (20 molecules/split, 2 epochs by default). Run this first, especially the first time against a new `.pkl` — it exercises the full pipeline (data -> PhysChem -> CAL -> loss -> eval) without paying for a real training run.

Note: `--smoke-test` truncates molecule *count* per split, not molecule *size* — this dataset includes some large peptides (500+ atoms), so even the smoke test can take a few minutes on CPU; it will be much faster once this is actually running on the GPU runtime.

In [ ]:
!python main.py --pkl-path "$PKL_PATH" --smoke-test

## 8. Full training run (PhysChem + CAL only)

The real run: trains for `--epochs` epochs, saves the best-validation-RMSE checkpoint to `checkpoints/best_model.pt`, and evaluates on the OOD test split at the end. Adjust `--epochs` / `--batch-size` / `--accumulation-steps` as needed for your GPU.

In [ ]:
!python main.py --pkl-path "$PKL_PATH" --epochs 100 --batch-size 8

## 9. (Optional, expensive) Phase-3 counterfactual explanations

Only run this when you actually want counterfactuals — this is the costly part of the pipeline. `--skip-train` reuses the checkpoint saved in step 8 instead of retraining. Pass `--query-smiles "<smiles>" "<smiles>" ...` to explain specific molecules instead of randomly sampled OOD-test ones.

In [ ]:
!python main.py --pkl-path "$PKL_PATH" \
    --checkpoint checkpoints/best_model.pt --skip-train \
    --phase3 --n-queries 3

## 10. Commit results back to git

Every run appends its config + metrics to `results/<today>.md`. Set your git identity once per Colab session, then commit and push.

In [ ]:
!git config --global user.email "you@example.com"   # TODO: fill in
!git config --global user.name "Your Name"           # TODO: fill in

In [ ]:
# Only needed if the repo is private: generate a GitHub Personal Access Token
# (github.com -> Settings -> Developer settings -> Fine-grained tokens) and paste it below.
# Leave blank if the repo is public or this runtime is already authenticated.
import getpass
token = getpass.getpass("GitHub token (leave blank if not needed): ")
if token:
    remote_url = REPO_URL.replace("https://", f"https://{token}@")
    !git remote set-url origin "$remote_url"

In [ ]:
!git add results/
!git commit -m "Colab run results: $(date +%Y-%m-%d)"
!git push